In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from config.settings import *
from src.preprocessing.scaler import FeatureScaler, DimensionalityReducer
from src.clustering.optimizer import ClusterOptimizer
from src.clustering.kmeans_clusterer import KMeansClusterer
from src.clustering.dbscan_clusterer import DBSCANClusterer
from src.clustering.hdbscan_clusterer import HDBSCANClusterer
from src.clustering.hierarchical_clusterer import HierarchicalClusterer
from src.clustering.gmm_clusterer import GMMClusterer
from src.clustering.evaluator import ClusterEvaluator
from src.clustering.comparison import ClusterComparison
from src.insights.cluster_profiler import ClusterProfiler
from src.visualization.cluster_plots import ClusterVisualizer
from src.visualization.map_plots import MapVisualizer

print("✅ All imports loaded")

In [ ]:
# Load climate normals
normals = pd.read_csv(PROCESSED_DIR / "climate_normals.csv")
print(f"📊 Shape: {normals.shape}")
print(f"📍 Points: {len(normals)}")

# Select features
exclude = ["latitude", "longitude", "date", "year", "month", "season", "days_count"]
feature_cols = [
    c for c in normals.select_dtypes(include=[np.number]).columns
    if c not in exclude
]
print(f"📋 Features: {len(feature_cols)}")

# Scale
scaler = FeatureScaler(method="standard")
X_scaled = scaler.fit_transform(normals, feature_cols)

# PCA
reducer = DimensionalityReducer()
X_pca = reducer.fit_pca(X_scaled, n_components=0.95, feature_names=feature_cols)

# Plot explained variance
variance = reducer.get_explained_variance()
cumulative = np.cumsum(variance)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.bar(range(1, len(variance) + 1), variance, alpha=0.7)
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Variance Explained")
ax1.set_title("Individual Variance")

ax2.plot(range(1, len(cumulative) + 1), cumulative, "ro-")
ax2.axhline(y=0.95, color="gray", linestyle="--", label="95% threshold")
ax2.set_xlabel("Number of Components")
ax2.set_ylabel("Cumulative Variance")
ax2.set_title("Cumulative Variance")
ax2.legend()

plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "pca_variance.png"), dpi=150)
plt.show()

In [ ]:
optimizer = ClusterOptimizer(k_range=range(3, 16))

# Elbow + Silhouette
sil_results = optimizer.silhouette_analysis(X_pca)

viz = ClusterVisualizer()
viz.plot_elbow(sil_results)

# GMM BIC/AIC
bic_results = optimizer.gmm_bic_aic(X_pca)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(bic_results["k"], bic_results["bic"], "bo-", label="BIC")
ax.plot(bic_results["k"], bic_results["aic"], "rs-", label="AIC")
ax.set_xlabel("Number of Components (K)")
ax.set_ylabel("Score")
ax.set_title("GMM Model Selection (BIC/AIC)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "gmm_bic_aic.png"), dpi=150)
plt.show()

# Optimal K
k_sil = optimizer.find_optimal_k(X_pca, method="silhouette")
k_bic = optimizer.find_optimal_k(X_pca, method="bic")
print(f"\n🎯 Optimal K (Silhouette): {k_sil}")
print(f"🎯 Optimal K (BIC):        {k_bic}")
optimal_k = k_sil
print(f"🎯 Using K = {optimal_k}")

In [ ]:
# DBSCAN eps estimation
k_distances = optimizer.estimate_dbscan_eps(X_pca)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(k_distances, linewidth=0.5)
ax.set_xlabel("Points (sorted)")
ax.set_ylabel("5th Nearest Neighbor Distance")
ax.set_title("DBSCAN Epsilon Estimation")
ax.grid(True, alpha=0.3)

# Mark suggested eps
eps_90 = np.percentile(k_distances, 90)
ax.axhline(y=eps_90, color="red", linestyle="--", label=f"90th percentile: {eps_90:.3f}")
ax.legend()
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "dbscan_eps.png"), dpi=150)
plt.show()

# Run ALL algorithms
comparison = ClusterComparison(X_pca)
results = comparison.run_all(
    optimal_k=optimal_k,
    dbscan_eps=eps_90,
    dbscan_min_samples=5,
    hdbscan_min_cluster_size=max(10, len(normals) // 50),
)

In [ ]:
comparison_df = comparison.get_comparison()
print("\n📊 Algorithm Comparison:")
print(comparison_df.to_string(index=False))

# Visual comparison
viz.plot_comparison(comparison_df)

best = comparison.get_best_algorithm()
print(f"\n🏆 Best algorithm: {best}")

In [ ]:
all_labels = comparison.get_all_labels()

fig, axes = plt.subplots(2, 3, figsize=(20, 13))
axes = axes.flatten()

from sklearn.decomposition import PCA
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_pca)

for idx, (name, labels) in enumerate(all_labels.items()):
    if idx >= 6:
        break
    ax = axes[idx]

    unique = np.unique(labels)
    for label in unique:
        mask = labels == label
        marker = "x" if label == -1 else "o"
        alpha = 0.3 if label == -1 else 0.6
        name_label = "Noise" if label == -1 else f"C{label}"
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1], s=15, alpha=alpha, marker=marker, label=name_label)

    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.grid(True, alpha=0.2)

    n_clusters = len([l for l in unique if l >= 0])
    n_noise = (labels == -1).sum()
    ax.text(0.02, 0.98, f"K={n_clusters}, Noise={n_noise}",
            transform=ax.transAxes, fontsize=8, va="top",
            bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

# Hide unused axes
for idx in range(len(all_labels), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle("All Clustering Results (PCA 2D)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "all_algorithms_2d.png"), dpi=150)
plt.show()

In [ ]:
best_labels = normals.get("cluster", all_labels.get("kmeans", list(all_labels.values())[0]))
normals["cluster"] = all_labels.get("kmeans", list(all_labels.values())[0])

viz.plot_silhouette_analysis(X_pca, normals["cluster"].values, title="K-Means")

In [ ]:
profiler = ClusterProfiler()

# Summary profile
profile_summary = profiler.profile_summary(normals, feature_cols, "cluster")
print("📊 Cluster Profiles (mean values):")
print(profile_summary[["cluster_size", "cluster_pct"]].to_string())

# Auto labels
auto_labels = profiler.auto_label_clusters(normals, feature_cols, "cluster")
print(f"\n🏷️  Auto Labels:")
for k, v in auto_labels.items():
    print(f"   Cluster {k}: {v}")

# Detailed descriptions
for cid in sorted(normals["cluster"].unique()):
    if cid < 0:
        continue
    desc = profiler.describe_cluster(normals, cid, feature_cols, "cluster")
    print(f"\n{'='*50}")
    print(f"Cluster {cid}: {auto_labels.get(cid, '')} ({desc['size']} points, {desc['pct']}%)")
    print(f"{'='*50}")
    for char in desc["characteristics"][:8]:
        bar = "█" * int(abs(char["z_score"]) * 5)
        direction = "↑" if char["z_score"] > 0 else "↓"
        print(f"  {direction} {char['feature']:<40s} {char['cluster_mean']:>8.2f}  (global: {char['global_mean']:.2f})  {bar}")

In [ ]:
# Pick top 8-10 most important features for radar
important_features = [
    "temperature_2m_mean", "precipitation_sum",
    "relative_humidity_2m_mean", "wind_speed_10m_mean",
    "cloud_cover_mean", "pressure_msl_mean",
    "shortwave_radiation_mean", "et0_fao_evapotranspiration_sum",
]
available = [f for f in important_features if f in profile_summary.columns]

viz.plot_radar_profile(profile_summary, available, cluster_labels=auto_labels)

In [ ]:
mapper = MapVisualizer()

# Cluster map
cluster_map = mapper.plot_cluster_map(
    normals,
    label_col="cluster",
    title="India Weather Pattern Clusters",
    cluster_labels=auto_labels,
)
cluster_map

In [ ]:
# Temperature heatmap
if "temperature_2m_mean" in normals.columns:
    temp_map = mapper.plot_feature_heatmap(normals, "temperature_2m_mean", "Mean Temperature")

# Precipitation heatmap
if "precipitation_sum" in normals.columns:
    rain_map = mapper.plot_feature_heatmap(normals, "precipitation_sum", "Mean Precipitation")

# Wind heatmap
if "wind_speed_10m_mean" in normals.columns:
    wind_map = mapper.plot_feature_heatmap(normals, "wind_speed_10m_mean", "Mean Wind Speed")

print("✅ All maps saved to outputs/figures/")

In [ ]:
# Save clustered data
normals.to_csv(PROCESSED_DIR / "climate_normals_clustered.csv", index=False)

# Save comparison
comparison_df.to_csv(TABLES_DIR / "algorithm_comparison.csv", index=False)
profile_summary.to_csv(TABLES_DIR / "cluster_profiles.csv")

# Save models
for name, clusterer in results.items():
    clusterer.save(MODELS_DIR / f"{name}_model.pkl")

print("=" * 65)
print("  ✅ CLUSTERING COMPLETE!")
print("=" * 65)
print(f"  Points:     {len(normals)}")
print(f"  Features:   {len(feature_cols)}")
print(f"  Optimal K:  {optimal_k}")
print(f"  Best algo:  {best}")
print(f"\n  Clusters:")
for cid, label in auto_labels.items():
    size = (normals["cluster"] == cid).sum()
    print(f"    {cid}: {label} ({size} pts)")
print("=" * 65)